# CRM Customer Churn Intelligence

???????????????? notebook ??? ???: ???????? ????????, ???????? ?????? ? ?????????? `model.joblib`.


In [ ]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [ ]:
dataset_path = Path('../data/crm_customer_churn.csv')
df = pd.read_csv(dataset_path)
df.head()


In [ ]:
numeric_columns = ['senior_citizen', 'tenure_months', 'monthly_charges', 'total_charges', 'service_count']
categorical_columns = ['contract_type', 'payment_method', 'internet_service', 'paperless_billing', 'has_family_plan', 'has_tech_support']
df['lifecycle_stage'] = pd.cut(
    df['tenure_months'],
    bins=[-1, 6, 24, 48, 999],
    labels=['Onboarding', 'Growth', 'Retention', 'Loyal']
).astype(str)
categorical_columns.append('lifecycle_stage')
X = df.drop(columns=['customer_id', 'churn_target'])
y = df['churn_target']
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_columns),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_columns),
])
model = RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=4, random_state=42, class_weight='balanced_subsample')
pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
pipeline.fit(X_train, y_train)
pred = pipeline.predict(X_test)
prob = pipeline.predict_proba(X_test)[:, 1]
metrics = {
    'accuracy': accuracy_score(y_test, pred),
    'precision': precision_score(y_test, pred),
    'recall': recall_score(y_test, pred),
    'f1_score': f1_score(y_test, pred),
    'roc_auc': roc_auc_score(y_test, prob),
}
metrics


In [ ]:
artifact = {'pipeline': pipeline, 'metadata': {'metrics': metrics, 'dataset': str(dataset_path)}}
joblib.dump(artifact, '../model.joblib')
Path('../model.joblib').resolve()
